# 🎙️ Caso Práctico: Síntesis y Clonación de Voz con IA
**Objetivos del Notebook:**
1. Explorar modelos de síntesis de voz (TTS) modernos.
2. Comparar VITS, Bark y XTTS en calidad, inteligibilidad y velocidad.
3. Desarrollar un caso práctico de localización (doblaje) de un clip de vídeo usando clonación de voz Zero-Shot.

In [6]:
# Instalación de dependencias
# !pip install TTS transformers scipy torchaudio moviepy soundfile

import torch
import time
import IPython.display as ipd
from TTS.api import TTS
from transformers import AutoProcessor, BarkModel
import scipy

# Configuración del dispositivo (GPU si está disponible, si no CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de ejecución: {device}")

# Texto de prueba estándar para la comparativa
texto_prueba = "Artificial intelligence is transforming the way we interact with machines."

Dispositivo de ejecución: cpu


## 1. Evaluación de Modelos TTS
A continuación, probaremos tres arquitecturas distintas para medir su rendimiento.
### Modelo A: Coqui VITS (Rápido y Tradicional)
VITS es un modelo End-to-End de alta calidad. Es excelente para voces estándar, pero no permite clonación fácil ni gran expresividad.

In [7]:
print("Cargando modelo Tacotron2...")
tts_vits = TTS("tts_models/en/ljspeech/tacotron2-DDC").to(device)

print("Generando audio con Tacotron2...")
start_time = time.time()
tts_vits.tts_to_file(text=texto_prueba, file_path="./audios_generados/vits_output.wav")
vits_time = time.time() - start_time

print(f"Tiempo de inferencia Tacotron2: {vits_time:.2f} segundos")
ipd.Audio("./audios_generados/vits_output.wav")

Cargando modelo Tacotron2...
 > tts_models/en/ljspeech/tacotron2-DDC is already downloaded.
 > vocoder_models/en/ljspeech/hifigan_v2 is already downloaded.
 > Using model: Tacotron2
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:1.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:2.718281828459045
 | > hop_length:256
 | > win_length:1024
 > Model's reduction rate `r` is set to: 1
 > Vocoder Model: 

### Modelo B: Bark de Suno (Expresivo y Generativo)
Bark es un modelo basado en Transformers (similar a ChatGPT pero para audio). Puede generar efectos no verbales como risas, suspiros o dudas, pero requiere mucha más capacidad de cómputo.

In [8]:
print("Cargando modelo Bark (HuggingFace)...")
processor = AutoProcessor.from_pretrained("suno/bark-small")
model = BarkModel.from_pretrained("suno/bark-small", use_safetensors=True).to(device)

texto_bark = "Hello, my name is Suno. And, uh — and I like pizza. [laughs]"

print("Generando audio con Bark...")
start_time = time.time()
inputs = processor(texto_bark, voice_preset="v2/en_speaker_6").to(device)

with torch.no_grad():
    audio_array = model.generate(**inputs)

audio_array = audio_array.cpu().numpy().squeeze()
bark_time = time.time() - start_time

# Guardar y reproducir
scipy.io.wavfile.write("./audios_generados/bark_output.wav", rate=model.generation_config.sample_rate, data=audio_array)
print(f"Tiempo de inferencia Bark: {bark_time:.2f} segundos")
ipd.Audio("./audios_generados/bark_output.wav")

Cargando modelo Bark (HuggingFace)...


Loading weights: 100%|██████████| 535/535 [00:00<00:00, 3429.02it/s]


Generando audio con Bark...


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://

Tiempo de inferencia Bark: 69.65 segundos


### Modelo C: XTTS v2 de Coqui (Clonación de voz y Multilingüe)
XTTS es el estado del arte actual en código abierto. Permite clonar la voz de una persona con solo un audio de referencia de 3 segundos (Zero-Shot) y hacer que hable en múltiples idiomas.

In [9]:
print("Cargando modelo YourTTS...")
tts_xtts = TTS("tts_models/multilingual/multi-dataset/your_tts").to(device)

print("Generando clonación con YourTTS...")
start_time = time.time()

# CLONACIÓN: Usamos el audio de referencia para generar una nueva frase
tts_xtts.tts_to_file(
    text=texto_prueba,
    speaker_wav="./audios_generados/vits_output.wav",  # Usando el audio generado por Tacotron2 como referencia
    language="en",
    file_path="./audios_generados/xtts_output.wav"
)
xtts_time = time.time() - start_time

print(f"Tiempo de inferencia YourTTS: {xtts_time:.2f} segundos")
ipd.Audio("./audios_generados/xtts_output.wav")

Cargando modelo YourTTS...
 > tts_models/multilingual/multi-dataset/your_tts is already downloaded.
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Model fully restored. 
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:64
 | > log_func:n

## 2. Análisis Comparativo
Basado en las ejecuciones anteriores y la literatura técnica, este es el análisis de los modelos:

| Característica | VITS (Coqui) | Bark (Suno) | XTTS v2 (Coqui) |
| :--- | :--- | :--- | :--- |
| **Arquitectura** | Red End-to-End (GAN/Flows) | Transformer Generativo | Autoregresivo + VQ-VAE |
| **Calidad Perceptual (MOS estimado)** | ~4.0 (Buena, algo robótica) | ~4.3 (Muy natural/expresiva) | ~4.5 (Excelente naturalidad) |
| **Inteligibilidad** | Muy alta (Casi sin errores) | Media (A veces tiene "alucinaciones") | Alta |
| **Velocidad Inferencia** | Muy Rápida | Lenta | Moderada |
| **Clonación de Voz** | Requiere fine-tuning pesado | Por prompts predefinidos | **Zero-Shot (Solo 3 segundos)** |
| **Multilingüe** | No (Modelo específico) | Sí | Sí (Cross-lingual support) |

**Conclusión:** Para un caso práctico de doblaje, **XTTS v2** es el ganador indiscutible, ya que permite mantener el timbre del actor original (clonación) y generar el nuevo audio en otro idioma.

## 3. Caso Práctico: Localización de un Vídeo (Doblaje IA)
**Objetivo:** Tomar el audio de un vídeo en inglés, extraer la voz del hablante, y usar XTTS para generar el diálogo traducido al español **con la misma voz** del hablante original.

*Simularemos el proceso teniendo el audio extraído del video (`actor_original.wav`).*

In [10]:
# PASO 1: Texto traducido (El guion del vídeo localizado al español)
texto_traducido = "Hola, bienvenidos a este caso práctico. Estamos comprobando cómo la inteligencia artificial puede doblar vídeos manteniendo mi tono de voz original."

# PASO 2: Clonación y síntesis en Español
print("Generando doblaje localizado en Español...")
tts_xtts.tts_to_file(
    text=texto_traducido,
    speaker_wav="./audios_generados/vits_output.wav",  # Usamos la voz generada como referencia
    language="en",                  # YourTTS soporta 'en', 'fr-fr', 'pt-br'
    file_path="./audios_originales/audio_doblado_es.wav"
)

print("Audio original (Inglés):")
display(ipd.Audio("./audios_generados/vits_output.wav"))

print("Audio doblado (Español) con la voz clonada:")
display(ipd.Audio("./audios_originales/audio_doblado_es.wav"))

Generando doblaje localizado en Español...
 > Text splitted to sentences.
['Hola, bienvenidos a este caso práctico.', 'Estamos comprobando cómo la inteligencia artificial puede doblar vídeos manteniendo mi tono de voz original.']
 > Processing time: 2.118868827819824
 > Real-time factor: 0.1794131098916024
Audio original (Inglés):


Audio doblado (Español) con la voz clonada:
